# Regression for time series data: lab notebook

**Statistical Modelling and Forecasting, Regression and exponential smoothing**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_ljungbox

### 1. Monthly CO2: trend, then trend + seasonal dummies

In [ ]:
# 1. Monthly CO2: trend, then trend + seasonal dummies
y = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate()
y = y["1985":]
n = len(y)
t = np.arange(1, n + 1)
month = pd.get_dummies(y.index.month, prefix="m", drop_first=True).astype(float)
D1 = sm.add_constant(pd.DataFrame({"t": t}, index=y.index))
D2 = pd.concat([D1, month.set_index(y.index)], axis=1)
train, test = y.iloc[:-12], y.iloc[-12:]
m1 = sm.OLS(train, D1.iloc[:-12]).fit()
m2 = sm.OLS(train, D2.iloc[:-12]).fit()
for name, m in [("trend only", m1), ("trend + 11 dummies", m2)]:
    print(f"{name:20s} R2 = {m.rsquared:.4f}  sigma = {np.sqrt(m.mse_resid):.3f}  Durbin-Watson = {durbin_watson(m.resid):.2f}  AIC = {m.aic:.1f}")
print("trend per year (ppm):", round(12 * m2.params["t"], 3), "| July effect against January:", round(m2.params["m_7"], 2))

### 2. Residual autocorrelation and the lagged-y model

In [ ]:
# 2. Residual autocorrelation and the lagged-y model
lb = acorr_ljungbox(m2.resid, lags=[12], return_df=True)
print("Ljung-Box p-value for the seasonal model's residuals: %.1e -> the residuals are still strongly autocorrelated" % float(lb["lb_pvalue"].iloc[0]))
D3 = D2.copy(); D3["y_lag"] = y.shift(1)
D3 = D3.dropna(); y3 = y.loc[D3.index]
m3 = sm.OLS(y3.iloc[:-12], D3.iloc[:-12]).fit()
print("with lagged y: coefficient on y(t-1) =", round(m3.params["y_lag"], 3), "| Durbin-Watson =", round(durbin_watson(m3.resid), 2))

### 3. Twelve-month forecasts with 95% prediction intervals (models 1 and 2)

In [ ]:
# 3. Twelve-month forecasts with 95% prediction intervals (models 1 and 2)
pr2 = m2.get_prediction(D2.iloc[-12:]).summary_frame(alpha=0.05)
rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
print("hold-out RMSE: trend only", round(rmse(test - m1.predict(D1.iloc[-12:])), 3), "| seasonal dummies", round(rmse(test - pr2["mean"].values), 3))
print("first 3 forecasts with 95% PI:")
print(pr2[["mean", "obs_ci_lower", "obs_ci_upper"]].head(3).round(2).to_string())

### 4. Spurious regression: two independent trending series

In [ ]:
# 4. Spurious regression: two independent trending series
rng = np.random.default_rng(1)
a = np.cumsum(rng.standard_normal(200)); b = np.cumsum(rng.standard_normal(200))
sp = sm.OLS(b, sm.add_constant(a)).fit()
print("independent random walks: R2 =", round(sp.rsquared, 2), "| t =", round(sp.tvalues[1], 1), "| DW =", round(durbin_watson(sp.resid), 2))
d = sm.OLS(np.diff(b), sm.add_constant(np.diff(a))).fit()
print("after differencing: R2 =", round(d.rsquared, 3), "| t =", round(d.tvalues[1], 2))

### 5. The numerical problem: quarterly regression and the lagged model

In [ ]:
# 5. The numerical problem: quarterly regression and the lagged model
a0, b0, g2, g3, g4 = 50, 2, 6, 12, -3
f = lambda tt, q: a0 + b0 * tt + {1: 0, 2: g2, 3: g3, 4: g4}[q]
print("forecasts t = 17, 18, 20:", f(17, 1), f(18, 2), f(20, 4))
print("lagged model forecast:", 4 + 0.6 * 40 + 3 * 8, "| long-run multiplier:", 3 / (1 - 0.6))

## Expected output

Running all cells above should print:

```
trend only           R2 = 0.9069  sigma = 2.259  Durbin-Watson = 0.31  AIC = 859.7
trend + 11 dummies   R2 = 0.9909  sigma = 0.728  Durbin-Watson = 0.17  AIC = 435.6
trend per year (ppm): 1.537 | July effect against January: 0.57
Ljung-Box p-value for the seasonal model's residuals: 1.7e-262 -> the residuals are still strongly autocorrelated
with lagged y: coefficient on y(t-1) = 0.918 | Durbin-Watson = 2.41
hold-out RMSE: trend only 2.146 | seasonal dummies 0.391
first 3 forecasts with 95% PI:
              mean  obs_ci_lower  obs_ci_upper
2001-01-01  369.89        368.39        371.38
2001-02-01  370.65        369.16        372.15
2001-03-01  371.62        370.13        373.11
independent random walks: R2 = 0.73 | t = 23.0 | DW = 0.13
after differencing: R2 = 0.001 | t = 0.51
forecasts t = 17, 18, 20: 84 92 87
lagged model forecast: 52.0 | long-run multiplier: 7.5
```

## Practice exercises

Try these in the cells below.

1. Fit "Trend only" to the seasonal series and look at the residual ACF. Then add seasonal dummies.
2. Increase the error autocorrelation to 0.8 and watch DW. Add the lagged y term to see if DW improves.
3. Compare the test-period RMSE of the three models.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
